# Phase 2 — Frozen Sentence-Embedding Retrieval Benchmark

This notebook evaluates a **zero-shot, frozen sentence-embedding baseline** for
BugHub Mozilla Firefox and Mozilla Core. It intentionally reuses the data gate,
temporal ground truth, split, and ranking metrics fixed in Phase 1, but it does
**not** rerun TF-IDF.

Protocol:

- validation queries are the latest 15% of each official training split;
- every query retrieves only reports created strictly earlier than the query;
- all known historical duplicates remain relevant (multi-relevant retrieval);
- embeddings are L2-normalized, so inner product is cosine similarity;
- official test is disabled until the embedding choice is frozen on validation.

The default run builds/loads Firefox embeddings and evaluates a deterministic
100-query smoke sample. Read the run switches before enabling full validation or
official test.


In [ ]:
import gc
import hashlib
import html
import importlib.metadata
import json
import math
import platform
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
    import sentence_transformers
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("Installing sentence-transformers (and its PyTorch dependency if needed)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"
    ])
    import torch
    import sentence_transformers
    from sentence_transformers import SentenceTransformer

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Paths, model, and run switches

The data paths match Phase 1. Embeddings are cached by project, model, issue
order, and preprocessed text fingerprint. A smoke run still has to encode the
project once on the first run, but later runs reuse the cache.

Safe default:

- `RUN_SMOKE_TEST = True`: Firefox, 100 deterministic validation queries;
- `RUN_FULL_VALIDATION = False`: do not launch the complete benchmark yet;
- `RUN_OFFICIAL_TEST = False`: keep the official test sealed.

After full validation, copy the automatically selected model name and pinned
revision into `SELECTED_TEST_MODEL_NAME` / `SELECTED_TEST_MODEL_REVISION`, then
enable the official test cell.


In [ ]:
COLAB_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/DuplicateDetection")
LOCAL_ROOT = Path(r"D:\Github\bughub")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

DATA_ROOT = COLAB_ROOT if COLAB_ROOT.exists() else LOCAL_ROOT
OUTPUT_ROOT = DATA_ROOT / "phase2_embedding_outputs"
CACHE_ROOT = OUTPUT_ROOT / "embedding_cache"
PHASE1_OUTPUT_ROOT = DATA_ROOT / "phase1_fixed_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

PROJECTS = {
    "Firefox": {
        "main": DATA_ROOT / "Firefox" / "mozilla_firefox.csv",
        "train": DATA_ROOT / "Firefox" / "train.csv",
        "test": DATA_ROOT / "Firefox" / "test.csv",
    },
    "MozillaCore": {
        "main": DATA_ROOT / "MozillaCore" / "mozilla_core.csv",
        "train": DATA_ROOT / "MozillaCore" / "train.csv",
        "test": DATA_ROOT / "MozillaCore" / "test.csv",
    },
}

VALIDATION_FRACTION_OF_OFFICIAL_TRAIN = 0.15
TOP_KS = (1, 5, 10)

# Frozen zero-shot semantic baseline. The revision is a full Hugging Face commit
# SHA, so upstream changes under the same model ID cannot silently change a run.
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
EMBEDDING_ENCODING_MODE = "symmetric"  # "symmetric" or "query_document"
QUERY_PREFIX = ""
DOCUMENT_PREFIX = ""
SELECTED_TEST_MODEL_NAME = None  # Fill only after inspecting full validation.
SELECTED_TEST_MODEL_REVISION = None

# Pre-registered selection rule. Aggregate projects equally, maximize Hit@10,
# then MRR@10 and Recall@10. candidate_id/model name is the deterministic tie.
SELECTION_AGGREGATION = "macro_project_average"
PRIMARY_SELECTION_METRIC = "hit@10"
SELECTION_TIE_BREAKERS = ("mrr@10", "recall@10")

# Put the title first so tokenizer truncation cannot remove it. Very long
# descriptions are capped before tokenization for stable memory and runtime.
DESCRIPTION_CHAR_LIMIT = 4_000
EMBEDDING_TEXT_VERSION = f"title-newline-description-head-{DESCRIPTION_CHAR_LIMIT}"
ENCODE_BATCH_SIZE = 128
EVAL_QUERY_BATCH_SIZE = 64
EVAL_DEVICE = None  # None -> CUDA when available, otherwise CPU.
TRUNCATION_AUDIT_SAMPLE = 2_000
TRUNCATION_WARNING_THRESHOLD_PCT = 20.0
FORCE_REBUILD_EMBEDDINGS = False

# Run control. Start with smoke, then full validation, then official test once.
RUN_SMOKE_TEST = True
SMOKE_PROJECT = "Firefox"
SMOKE_MAX_QUERIES = 100
RUN_FULL_VALIDATION = False
MAX_VALIDATION_QUERIES = None
RUN_OFFICIAL_TEST = False
MAX_TEST_QUERIES = None

for project, paths in PROJECTS.items():
    for kind, path in paths.items():
        assert path.exists(), f"Missing {project}/{kind}: {path}"

assert SMOKE_PROJECT in PROJECTS
assert DESCRIPTION_CHAR_LIMIT > 0
assert ENCODE_BATCH_SIZE > 0 and EVAL_QUERY_BATCH_SIZE > 0
assert re.fullmatch(r"[0-9a-f]{40}", EMBEDDING_MODEL_REVISION), (
    "Pin EMBEDDING_MODEL_REVISION to a full 40-character commit SHA."
)
assert EMBEDDING_ENCODING_MODE in {"symmetric", "query_document"}
if EMBEDDING_ENCODING_MODE == "symmetric":
    assert QUERY_PREFIX == DOCUMENT_PREFIX, (
        "Symmetric mode must use the same query/document prefix."
    )
assert PRIMARY_SELECTION_METRIC in {f"hit@{k}" for k in TOP_KS}
assert all(metric in {"mrr@10", "ndcg@10", *(f"recall@{k}" for k in TOP_KS)}
           for metric in SELECTION_TIE_BREAKERS)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Model:", EMBEDDING_MODEL_NAME)
print("Pinned revision:", EMBEDDING_MODEL_REVISION)
print("Encoding mode:", EMBEDDING_ENCODING_MODE)
print("Selection rule:", PRIMARY_SELECTION_METRIC, "then", SELECTION_TIE_BREAKERS)
print("Smoke/full-validation/official-test:",
      RUN_SMOKE_TEST, RUN_FULL_VALIDATION, RUN_OFFICIAL_TEST)


## 2. Load and normalize issues and official relevance files

`Duplicate` is a semicolon-separated multi-relevant set and is never reduced to
one target. Phase 2 keeps Phase 1's text normalization for identical data gates.
For the encoder, `embedding_text` contains the title once followed by the head of
the description; this avoids the Phase 1 TF-IDF title repetition and makes the
model's truncation policy explicit.


In [ ]:
def normalize_id(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return re.sub(r"\.0$", "", text)


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " <URL> ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_duplicate_set(value):
    if pd.isna(value):
        return frozenset()
    text = str(value).strip()
    if not text or text.upper() == "NULL":
        return frozenset()
    return frozenset(
        normalized
        for item in text.split(";")
        if (normalized := normalize_id(item))
    )


def load_project(project, paths):
    issues = pd.read_csv(paths["main"], low_memory=False)
    issues = issues.rename(columns={
        "Issue_id": "issue_id",
        "Priority": "priority",
        "Component": "component",
        "Duplicated_issue": "duplicate_of",
        "Title": "title",
        "Description": "description",
        "Resolution": "resolution",
        "Created_time": "created_at",
    })
    required = {
        "issue_id", "title", "description", "created_at",
        "duplicate_of", "component", "resolution",
    }
    assert required.issubset(issues.columns), required - set(issues.columns)

    issues["issue_id"] = issues["issue_id"].map(normalize_id)
    issues["created_at"] = pd.to_datetime(issues["created_at"], errors="coerce", utc=True)
    issues["title_clean"] = issues["title"].map(normalize_text)
    issues["description_clean"] = issues["description"].map(normalize_text)

    # Keep the Phase 1 field for identical non-empty-text filtering.
    issues["text"] = (
        issues["title_clean"] + " . " + issues["title_clean"]
        + " . " + issues["description_clean"]
    ).str.strip()

    description_head = issues["description_clean"].str.slice(0, DESCRIPTION_CHAR_LIMIT)
    issues["embedding_text"] = (
        issues["title_clean"] + "\n" + description_head
    ).str.strip()
    issues["description_char_truncated"] = (
        issues["description_clean"].str.len() > DESCRIPTION_CHAR_LIMIT
    )
    issues["project"] = project

    assert issues["issue_id"].notna().all()
    assert not issues["issue_id"].duplicated().any()
    assert issues["created_at"].notna().all()
    issues = issues.sort_values(["created_at", "issue_id"]).reset_index(drop=True)
    issues["position"] = np.arange(len(issues), dtype=np.int64)

    split_frames = {}
    for split in ("train", "test"):
        labels = pd.read_csv(paths[split], dtype=str)
        assert {"Issue_id", "Duplicate"}.issubset(labels.columns)
        labels = labels.rename(columns={"Issue_id": "issue_id", "Duplicate": "duplicate_raw"})
        labels["issue_id"] = labels["issue_id"].map(normalize_id)
        labels["relevant_ids"] = labels["duplicate_raw"].map(parse_duplicate_set)
        assert labels["issue_id"].notna().all()
        assert not labels["issue_id"].duplicated().any()
        split_frames[split] = labels[["issue_id", "relevant_ids"]].copy()

    all_ids = set(issues["issue_id"])
    train_ids = set(split_frames["train"]["issue_id"])
    test_ids = set(split_frames["test"]["issue_id"])
    assert train_ids.isdisjoint(test_ids), f"{project}: train/test ID overlap"
    assert train_ids | test_ids == all_ids, f"{project}: split does not cover main CSV"

    id_to_time = issues.set_index("issue_id")["created_at"].to_dict()
    id_to_text = issues.set_index("issue_id")["text"].to_dict()
    known = set(id_to_time)

    audit_rows = []
    for split, labels in split_frames.items():
        total_links = sum(len(x) for x in labels["relevant_ids"])
        missing = sum(
            target not in known
            for targets in labels["relevant_ids"]
            for target in targets
        )
        future = same = 0
        for query_id, targets in labels[["issue_id", "relevant_ids"]].itertuples(index=False):
            query_time = id_to_time[query_id]
            for target in targets:
                if target not in id_to_time:
                    continue
                future += id_to_time[target] > query_time
                same += id_to_time[target] == query_time
        audit_rows.append({
            "project": project,
            "split": split,
            "rows": len(labels),
            "labeled_queries": int((labels["relevant_ids"].map(len) > 0).sum()),
            "relevance_links": total_links,
            "missing_targets": missing,
            "future_targets": future,
            "same_time_targets": same,
            "description_char_truncated_pct": round(
                100 * issues["description_char_truncated"].mean(), 2
            ),
        })

    return {
        "project": project,
        "issues": issues,
        "train_labels": split_frames["train"],
        "test_labels": split_frames["test"],
        "id_to_time": id_to_time,
        "id_to_text": id_to_text,
        "audit": pd.DataFrame(audit_rows),
    }


datasets = {
    project: load_project(project, paths)
    for project, paths in PROJECTS.items()
}

data_audit = pd.concat([data["audit"] for data in datasets.values()], ignore_index=True)
display(data_audit)


## 3. Data-gate assertions

The official test files are suitable for rolling temporal retrieval only when
all labeled targets exist and none is from the future. Training label files may
contain future cluster members; validation construction filters those targets
at each query timestamp.


In [ ]:
dataset_summary = []
for project, data in datasets.items():
    issues = data["issues"]
    test_audit = data["audit"].query("split == 'test'").iloc[0]
    assert test_audit["missing_targets"] == 0
    assert test_audit["future_targets"] == 0
    assert not issues["issue_id"].duplicated().any()
    dataset_summary.append({
        "project": project,
        "issues": len(issues),
        "empty_text": int(issues["text"].str.len().eq(0).sum()),
        "created_min": issues["created_at"].min(),
        "created_max": issues["created_at"].max(),
        "official_train": len(data["train_labels"]),
        "official_test": len(data["test_labels"]),
        "test_labeled_queries": int(test_audit["labeled_queries"]),
        "test_relevance_links": int(test_audit["relevance_links"]),
    })

dataset_summary = pd.DataFrame(dataset_summary)
display(dataset_summary)
print("DATA GATE: PASS")


# Split-level leakage guard. Validation may use only official-train candidates.
# The current BugHub files are chronological, but the assertion makes that
# assumption executable instead of relying on the README or filename.
split_chronology_rows = []
for project, data in datasets.items():
    issues = data["issues"]
    official_train_ids = set(data["train_labels"]["issue_id"])
    official_test_ids = set(data["test_labels"]["issue_id"])
    train_times = issues.loc[
        issues["issue_id"].isin(official_train_ids), "created_at"
    ]
    test_times = issues.loc[
        issues["issue_id"].isin(official_test_ids), "created_at"
    ]
    train_max = train_times.max()
    test_min = test_times.min()
    assert train_max < test_min, (
        f"{project}: official split is not strictly chronological: "
        f"train_max={train_max}, test_min={test_min}. "
        "Keep the explicit allowed-candidate mask enabled."
    )
    data["official_train_ids"] = official_train_ids
    data["official_test_ids"] = official_test_ids
    split_chronology_rows.append({
        "project": project,
        "train_min": train_times.min(),
        "train_max": train_max,
        "test_min": test_min,
        "test_max": test_times.max(),
        "strictly_chronological": True,
    })

split_chronology = pd.DataFrame(split_chronology_rows)
display(split_chronology)
print("SPLIT CHRONOLOGY GATE: PASS")


## 4. Construct validation and test query sets

For every query, relevant reports are filtered to IDs that:

- exist in the same project corpus;
- have usable text;
- were created strictly before the query.

Queries with no remaining historical relevant report are not evaluable and are
reported separately. This prevents future-target leakage.


In [ ]:
def filter_temporal_queries(data, labels, allowed_query_ids=None):
    id_to_time = data["id_to_time"]
    id_to_text = data["id_to_text"]
    records = []
    audit = {
        "labeled_queries": 0,
        "empty_query_text": 0,
        "no_historical_relevant": 0,
        "future_or_same_relevant_removed": 0,
        "missing_or_empty_relevant_removed": 0,
    }
    allowed_query_ids = None if allowed_query_ids is None else set(allowed_query_ids)

    for query_id, raw_relevant in labels[["issue_id", "relevant_ids"]].itertuples(index=False):
        if allowed_query_ids is not None and query_id not in allowed_query_ids:
            continue
        if not raw_relevant:
            continue
        audit["labeled_queries"] += 1
        if not id_to_text.get(query_id, ""):
            audit["empty_query_text"] += 1
            continue

        query_time = id_to_time[query_id]
        relevant = set()
        for target in raw_relevant:
            if target not in id_to_time or not id_to_text.get(target, ""):
                audit["missing_or_empty_relevant_removed"] += 1
                continue
            if id_to_time[target] >= query_time:
                audit["future_or_same_relevant_removed"] += 1
                continue
            relevant.add(target)

        if not relevant:
            audit["no_historical_relevant"] += 1
            continue
        records.append({
            "query_id": query_id,
            "query_time": query_time,
            "relevant_ids": frozenset(relevant),
            "num_relevant": len(relevant),
        })

    query_frame = pd.DataFrame(records).sort_values(["query_time", "query_id"]).reset_index(drop=True)
    audit["evaluable_queries"] = len(query_frame)
    audit["evaluable_relevance_links"] = int(query_frame["num_relevant"].sum()) if len(query_frame) else 0
    return query_frame, audit


query_sets = {}
query_audits = []
for project, data in datasets.items():
    issues = data["issues"]
    train_labels = data["train_labels"]
    official_train_ids = set(train_labels["issue_id"])
    train_issues = issues[issues["issue_id"].isin(official_train_ids)].sort_values(["created_at", "issue_id"])

    validation_size = math.ceil(len(train_issues) * VALIDATION_FRACTION_OF_OFFICIAL_TRAIN)
    validation_ids = set(train_issues.tail(validation_size)["issue_id"])
    dev_fit_ids = set(train_issues.iloc[:-validation_size]["issue_id"])

    validation_queries, validation_audit = filter_temporal_queries(
        data, train_labels, validation_ids
    )
    test_queries, test_audit = filter_temporal_queries(data, data["test_labels"])

    assert all(
        data["id_to_time"][target] < row.query_time
        for row in validation_queries.itertuples()
        for target in row.relevant_ids
    )
    assert all(
        data["id_to_time"][target] < row.query_time
        for row in test_queries.itertuples()
        for target in row.relevant_ids
    )

    query_sets[project] = {
        "dev_fit_ids": dev_fit_ids,
        "official_train_ids": official_train_ids,
        "validation": validation_queries,
        "test": test_queries,
    }
    for split, audit in (("validation", validation_audit), ("test", test_audit)):
        query_audits.append({"project": project, "split": split, **audit})

query_audits = pd.DataFrame(query_audits)
display(query_audits)
print("TEMPORAL GROUND TRUTH: PASS")


## 5. Metrics, embedding cache, and batched temporal retrieval

The model is never fine-tuned on the benchmark. Each report is encoded
independently and cached by pinned model revision, role, preprocessing, package
versions, issue order, and text. Validation additionally masks candidates to the
official-train split; time filtering alone is not trusted as a split boundary.

`SentenceTransformerBackend` exposes separate query/document paths. MiniLM uses
the symmetric path and shares one matrix. Retrieval-oriented models can switch
to `query_document` mode and configure role prefixes without changing evaluator
logic.

The evaluator computes exact cosine scores in query batches with PyTorch. For
each batch it scores only up to the latest legal candidate boundary, then applies
the exact boundary for every query before deterministic Top-K selection. This
keeps Phase 1's ranking protocol without the slow Python loop of one dense matrix
multiplication per query.


In [ ]:
def ranking_metrics(retrieved_ids, relevant_ids, ks=TOP_KS):
    relevant = set(relevant_ids)
    result = {}
    for k in ks:
        hits = len(set(retrieved_ids[:k]) & relevant)
        result[f"hit@{k}"] = float(hits > 0)
        result[f"recall@{k}"] = hits / len(relevant)

    first_rank = next(
        (rank for rank, issue_id in enumerate(retrieved_ids[:10], start=1)
         if issue_id in relevant),
        None,
    )
    result["mrr@10"] = 0.0 if first_rank is None else 1.0 / first_rank

    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, issue_id in enumerate(retrieved_ids[:10], start=1)
        if issue_id in relevant
    )
    ideal_hits = min(len(relevant), 10)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    result["ndcg@10"] = dcg / idcg if idcg else 0.0
    return result


def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k == 0:
        return np.array([], dtype=np.int64)
    if k == len(scores):
        candidates = np.arange(len(scores))
    else:
        candidates = np.argpartition(-scores, k - 1)[:k]
    # Deterministic: score descending, then chronological corpus position.
    order = np.lexsort((candidates, -scores[candidates]))
    return candidates[order]


def deterministic_query_sample(queries, max_queries):
    if max_queries is None or len(queries) <= max_queries:
        return queries.reset_index(drop=True)
    selected = np.linspace(0, len(queries) - 1, max_queries, dtype=int)
    return queries.iloc[selected].drop_duplicates("query_id").reset_index(drop=True)


def embedding_fingerprint(issues, model_name, model_revision, cache_role):
    digest = hashlib.sha256()
    cache_settings = {
        "model_name": model_name,
        "model_revision": model_revision,
        "encoding_mode": EMBEDDING_ENCODING_MODE,
        "cache_role": cache_role,
        "query_prefix": QUERY_PREFIX,
        "document_prefix": DOCUMENT_PREFIX,
        "embedding_text_version": EMBEDDING_TEXT_VERSION,
        "sentence_transformers_version": sentence_transformers.__version__,
        "torch_version": torch.__version__,
    }
    digest.update(json.dumps(cache_settings, sort_keys=True).encode("utf-8"))
    for issue_id, text in issues[["issue_id", "embedding_text"]].itertuples(index=False):
        digest.update(issue_id.encode("utf-8"))
        digest.update(b"\0")
        digest.update(text.encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()[:20]


def safe_model_slug(model_name):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", model_name).strip("_")


class SentenceTransformerBackend:
    """Role-aware adapter; symmetric MiniLM remains the zero-cost default."""

    def __init__(self, model_name, model_revision):
        if not model_name or not model_revision:
            raise ValueError("Both model_name and pinned model_revision are required")
        self.model_name = model_name
        self.model_revision = model_revision
        self.model = SentenceTransformer(model_name, revision=model_revision)

    def prepare_texts(self, texts, role):
        if role not in {"query", "document"}:
            raise ValueError(f"Unknown embedding role: {role}")
        prefix = QUERY_PREFIX if role == "query" else DOCUMENT_PREFIX
        return [f"{prefix}{text}" for text in texts] if prefix else texts

    def encode_prepared(self, prepared_texts, role):
        method = self.model.encode
        if EMBEDDING_ENCODING_MODE == "query_document":
            role_method = "encode_query" if role == "query" else "encode_document"
            method = getattr(self.model, role_method, self.model.encode)
        return method(
            prepared_texts,
            batch_size=ENCODE_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32, copy=False)


_embedding_backend = None


def get_embedding_backend():
    global _embedding_backend
    if _embedding_backend is None:
        _embedding_backend = SentenceTransformerBackend(
            EMBEDDING_MODEL_NAME, EMBEDDING_MODEL_REVISION
        )
    return _embedding_backend


def audit_model_truncation(model, texts, sample_size=2_000, batch_size=128):
    if not texts:
        return {"token_audit_sample": 0, "model_max_tokens": None,
                "token_truncated_pct_estimate": None}
    sample_size = min(sample_size, len(texts))
    positions = np.linspace(0, len(texts) - 1, sample_size, dtype=int)
    sample = [texts[position] for position in positions]
    lengths = []
    for start in range(0, len(sample), batch_size):
        encoded = model.tokenizer(
            sample[start:start + batch_size],
            add_special_tokens=True,
            padding=False,
            truncation=False,
        )
        lengths.extend(len(ids) for ids in encoded["input_ids"])
    max_tokens = int(model.max_seq_length)
    audit = {
        "token_audit_sample": len(lengths),
        "model_max_tokens": max_tokens,
        "token_length_median": float(np.median(lengths)),
        "token_length_p95": float(np.percentile(lengths, 95)),
        "truncation_warning_threshold_pct": TRUNCATION_WARNING_THRESHOLD_PCT,
        "token_truncated_pct_estimate": round(
            100 * np.mean(np.asarray(lengths) > max_tokens), 2
        ),
    }
    if audit["token_truncated_pct_estimate"] > TRUNCATION_WARNING_THRESHOLD_PCT:
        print(
            "WARNING: substantial embedding input truncation: "
            f"{audit['token_truncated_pct_estimate']:.2f}% > "
            f"{TRUNCATION_WARNING_THRESHOLD_PCT:.2f}% threshold"
        )
    return audit


def build_or_load_embeddings(data, role, force_rebuild=False):
    if role not in {"query", "document"}:
        raise ValueError(f"Unknown embedding role: {role}")
    issues = data["issues"]
    cache_role = "shared" if EMBEDDING_ENCODING_MODE == "symmetric" else role
    fingerprint = embedding_fingerprint(
        issues, EMBEDDING_MODEL_NAME, EMBEDDING_MODEL_REVISION, cache_role
    )
    stem = (
        f"{data['project'].lower()}__{safe_model_slug(EMBEDDING_MODEL_NAME)}"
        f"__{cache_role}__{fingerprint}"
    )
    matrix_path = CACHE_ROOT / f"{stem}.npy"
    metadata_path = CACHE_ROOT / f"{stem}.json"

    if matrix_path.exists() and metadata_path.exists() and not force_rebuild:
        embeddings = np.load(matrix_path)
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        assert embeddings.shape[0] == len(issues)
        assert embeddings.dtype == np.float32
        metadata["cache_hit"] = True
        print(f"CACHE HIT: {data['project']} -> {matrix_path.name}")
        return embeddings, metadata

    print(f"ENCODING: {data['project']} / {cache_role} ({len(issues):,} reports)")
    backend = get_embedding_backend()
    encode_role = "document" if cache_role == "shared" else role
    texts = backend.prepare_texts(issues["embedding_text"].tolist(), encode_role)
    truncation_metadata = audit_model_truncation(
        backend.model, texts, TRUNCATION_AUDIT_SAMPLE, ENCODE_BATCH_SIZE
    )
    started = time.time()
    embeddings = backend.encode_prepared(texts, encode_role)

    assert embeddings.ndim == 2 and embeddings.shape[0] == len(issues)
    norm_sample = np.linalg.norm(embeddings[:min(2_000, len(embeddings))], axis=1)
    nonzero = norm_sample > 0
    assert np.allclose(norm_sample[nonzero], 1.0, atol=1e-4)

    metadata = {
        "project": data["project"],
        "representation": "embedding",
        "model_name": EMBEDDING_MODEL_NAME,
        "model_revision": EMBEDDING_MODEL_REVISION,
        "encoding_mode": EMBEDDING_ENCODING_MODE,
        "cache_role": cache_role,
        "query_prefix": QUERY_PREFIX,
        "document_prefix": DOCUMENT_PREFIX,
        "embedding_text_version": EMBEDDING_TEXT_VERSION,
        "fingerprint": fingerprint,
        "reports": len(issues),
        "embedding_dim": int(embeddings.shape[1]),
        "dtype": str(embeddings.dtype),
        "encode_seconds": round(time.time() - started, 2),
        "sentence_transformers_version": sentence_transformers.__version__,
        "torch_version": torch.__version__,
        "cache_hit": False,
        **truncation_metadata,
    }
    np.save(matrix_path, embeddings)
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"CACHE SAVED: {matrix_path}")
    display(pd.DataFrame([metadata]))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return embeddings, metadata


def evaluate_dense_rolling(
    data,
    queries,
    document_embeddings,
    query_embeddings,
    model_name,
    model_revision,
    allowed_candidate_ids,
    max_queries=None,
    query_batch_size=64,
    device=None,
):
    issues = data["issues"]
    assert document_embeddings.shape[0] == len(issues)
    assert query_embeddings.shape[0] == len(issues)
    eval_queries = deterministic_query_sample(queries, max_queries)
    assert len(eval_queries) > 0

    issue_ids = issues["issue_id"].to_numpy()
    created_ns = issues["created_at"].astype("int64").to_numpy()
    id_to_position = dict(zip(issue_ids, issues["position"]))
    query_positions = np.asarray(
        [id_to_position[query_id] for query_id in eval_queries["query_id"]],
        dtype=np.int64,
    )
    query_times_ns = eval_queries["query_time"].astype("int64").to_numpy()
    candidate_ends = np.searchsorted(created_ns, query_times_ns, side="left")
    assert np.all(candidate_ends > 0)
    assert np.all(candidate_ends <= query_positions)

    allowed_candidate_ids = set(allowed_candidate_ids)
    allowed_mask = np.fromiter(
        (issue_id in allowed_candidate_ids for issue_id in issue_ids),
        dtype=bool,
        count=len(issue_ids),
    )
    assert allowed_mask.any()

    resolved_device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    corpus = torch.from_numpy(np.asarray(document_embeddings)).to(resolved_device)
    shared_query_document_matrix = query_embeddings is document_embeddings
    query_matrix = (
        corpus
        if shared_query_document_matrix
        else torch.from_numpy(np.asarray(query_embeddings)).to(resolved_device)
    )
    rows = []
    started = time.time()
    max_k = max(TOP_KS)

    with torch.inference_mode():
        for start in range(0, len(eval_queries), query_batch_size):
            stop = min(start + query_batch_size, len(eval_queries))
            batch_positions = query_positions[start:stop]
            batch_candidate_ends = candidate_ends[start:stop]
            batch_corpus_end = int(batch_candidate_ends.max())

            query_tensor = query_matrix[
                torch.as_tensor(batch_positions, dtype=torch.long, device=resolved_device)
            ]
            score_batch = (
                query_tensor @ corpus[:batch_corpus_end].T
            ).float().cpu().numpy()

            for offset, row in enumerate(
                eval_queries.iloc[start:stop].itertuples(index=False)
            ):
                candidate_end = int(batch_candidate_ends[offset])
                local_scores = score_batch[offset, :candidate_end]
                local_allowed = allowed_mask[:candidate_end]
                if local_allowed.all():
                    local_top = top_k_indices(local_scores, max_k)
                else:
                    legal_positions = np.flatnonzero(local_allowed)
                    assert len(legal_positions) > 0
                    legal_top = top_k_indices(local_scores[legal_positions], max_k)
                    local_top = legal_positions[legal_top]
                retrieved = issue_ids[local_top].tolist()

                assert row.query_id not in retrieved
                assert set(retrieved).issubset(allowed_candidate_ids)
                assert all(data["id_to_time"][item] < row.query_time for item in retrieved)
                metrics = ranking_metrics(retrieved, row.relevant_ids)
                rows.append({
                    "project": data["project"],
                    "representation": "embedding",
                    "model_name": model_name,
                    "model_revision": model_revision,
                    "encoding_mode": EMBEDDING_ENCODING_MODE,
                    "query_id": row.query_id,
                    "query_time": row.query_time,
                    "num_candidates": int(local_allowed.sum()),
                    "num_relevant": len(row.relevant_ids),
                    "retrieved_ids": ";".join(retrieved),
                    **metrics,
                })

            completed = stop
            if completed % 256 == 0 or completed == len(eval_queries):
                print(f"  {data['project']} / embedding: {completed}/{len(eval_queries)}")
            del query_tensor, score_batch

    elapsed = time.time() - started
    results = pd.DataFrame(rows)
    metric_columns = [
        *(f"hit@{k}" for k in TOP_KS),
        *(f"recall@{k}" for k in TOP_KS),
        "mrr@10", "ndcg@10",
    ]
    summary = {
        "project": data["project"],
        "representation": "embedding",
        "model_name": model_name,
        "model_revision": model_revision,
        "encoding_mode": EMBEDDING_ENCODING_MODE,
        "embedding_text_version": EMBEDDING_TEXT_VERSION,
        "queries": len(results),
        "median_candidates": int(results["num_candidates"].median()),
        "retrieval_seconds": round(elapsed, 2),
        "evaluation_device": resolved_device,
        "query_batch_size": query_batch_size,
        **{column: results[column].mean() for column in metric_columns},
    }

    if not shared_query_document_matrix:
        del query_matrix
    del corpus
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results, summary


# Metric and ranking unit tests.
metric_check = ranking_metrics(["x", "b", "y"], {"a", "b"}, ks=(1, 2, 3))
assert metric_check["hit@1"] == 0
assert metric_check["hit@2"] == 1
assert metric_check["recall@2"] == 0.5
assert metric_check["mrr@10"] == 0.5
assert top_k_indices(np.array([0.1, 0.9, 0.9], dtype=np.float32), 2).tolist() == [1, 2]
print("METRIC + TOP-K TESTS: PASS")


## 6. Lazy embedding store

Embeddings are built only when a project is actually evaluated. Consequently,
the default Firefox smoke test does not encode Mozilla Core. The first run can
take several minutes; subsequent runs should report `CACHE HIT`.


In [ ]:
embedding_store = {}
embedding_metadata = {}


def get_project_representations(project):
    key = (
        project,
        EMBEDDING_MODEL_NAME,
        EMBEDDING_MODEL_REVISION,
        EMBEDDING_ENCODING_MODE,
        QUERY_PREFIX,
        DOCUMENT_PREFIX,
    )
    if key not in embedding_store:
        document_embeddings, document_metadata = build_or_load_embeddings(
            data=datasets[project], role="document",
            force_rebuild=FORCE_REBUILD_EMBEDDINGS,
        )
        if EMBEDDING_ENCODING_MODE == "symmetric":
            query_embeddings = document_embeddings
            query_metadata = document_metadata
        else:
            query_embeddings, query_metadata = build_or_load_embeddings(
                data=datasets[project], role="query",
                force_rebuild=FORCE_REBUILD_EMBEDDINGS,
            )
        embedding_store[key] = {
            "documents": document_embeddings,
            "queries": query_embeddings,
        }
        embedding_metadata[(project, "document")] = document_metadata
        embedding_metadata[(project, "query")] = query_metadata
    return embedding_store[key]


print("Lazy store ready. No project has been encoded by this cell.")


## 7. Smoke test — run this first

This is a deterministic chronological spread of 100 Firefox validation queries,
not a final metric estimate. It verifies model download, embedding cache,
temporal masking, Top-K output, and runtime before a full run.


In [ ]:
smoke_results = pd.DataFrame()
smoke_summary = pd.DataFrame()

if RUN_SMOKE_TEST:
    smoke_representations = get_project_representations(SMOKE_PROJECT)
    smoke_results, one_smoke_summary = evaluate_dense_rolling(
        data=datasets[SMOKE_PROJECT],
        queries=query_sets[SMOKE_PROJECT]["validation"],
        document_embeddings=smoke_representations["documents"],
        query_embeddings=smoke_representations["queries"],
        model_name=EMBEDDING_MODEL_NAME,
        model_revision=EMBEDDING_MODEL_REVISION,
        allowed_candidate_ids=datasets[SMOKE_PROJECT]["official_train_ids"],
        max_queries=SMOKE_MAX_QUERIES,
        query_batch_size=EVAL_QUERY_BATCH_SIZE,
        device=EVAL_DEVICE,
    )
    smoke_summary = pd.DataFrame([one_smoke_summary])
    display(smoke_summary)
    smoke_results.to_csv(OUTPUT_ROOT / "firefox_validation_smoke_query_results.csv", index=False)
    smoke_summary.to_csv(OUTPUT_ROOT / "firefox_validation_smoke_summary.csv", index=False)
else:
    print("SMOKE SKIPPED: set RUN_SMOKE_TEST=True to run it.")


## 8. Full validation — model selection happens here

Enable only after the smoke test succeeds. This evaluates all temporally valid
validation queries for both projects when `MAX_VALIDATION_QUERIES = None`.
It does not touch official test labels beyond the earlier data-integrity gate.

If Phase 1's `validation_summary.csv` exists, it is displayed beside Phase 2 for
convenience; the files remain separate.


In [ ]:
validation_results = {}
validation_summary = pd.DataFrame()


def select_model_from_macro_rows(macro_rows):
    """Apply the pre-registered rule; works unchanged with multiple models."""
    ranking_columns = [PRIMARY_SELECTION_METRIC, *SELECTION_TIE_BREAKERS]
    required = {"model_name", "model_revision", *ranking_columns}
    assert required.issubset(macro_rows.columns), required - set(macro_rows.columns)
    ranked = macro_rows.sort_values(
        [*ranking_columns, "model_name", "model_revision"],
        ascending=([False] * len(ranking_columns)) + [True, True],
        kind="mergesort",
    ).reset_index(drop=True)
    return ranked.iloc[0], ranked


if RUN_FULL_VALIDATION:
    validation_summaries = []
    for project, data in datasets.items():
        print(f"\n=== EMBEDDING VALIDATION: {project} ===")
        representations = get_project_representations(project)
        results, summary = evaluate_dense_rolling(
            data=data,
            queries=query_sets[project]["validation"],
            document_embeddings=representations["documents"],
            query_embeddings=representations["queries"],
            model_name=EMBEDDING_MODEL_NAME,
            model_revision=EMBEDDING_MODEL_REVISION,
            allowed_candidate_ids=data["official_train_ids"],
            max_queries=MAX_VALIDATION_QUERIES,
            query_batch_size=EVAL_QUERY_BATCH_SIZE,
            device=EVAL_DEVICE,
        )
        validation_results[project] = results
        validation_summaries.append(summary)
        results.to_csv(
            OUTPUT_ROOT / f"{project.lower()}_validation_query_results.csv", index=False
        )
        display(pd.DataFrame([summary]))

    project_validation_summary = pd.DataFrame(validation_summaries)
    metric_columns = [
        *(f"hit@{k}" for k in TOP_KS),
        *(f"recall@{k}" for k in TOP_KS),
        "mrr@10", "ndcg@10",
    ]
    macro = {
        "project": "MACRO_PROJECT_AVERAGE",
        "representation": "embedding",
        "model_name": EMBEDDING_MODEL_NAME,
        "model_revision": EMBEDDING_MODEL_REVISION,
        "encoding_mode": EMBEDDING_ENCODING_MODE,
        "embedding_text_version": EMBEDDING_TEXT_VERSION,
        "queries": int(project_validation_summary["queries"].sum()),
        **{
            column: project_validation_summary[column].mean()
            for column in metric_columns
        },
    }
    validation_summary = pd.concat(
        [project_validation_summary, pd.DataFrame([macro])], ignore_index=True
    )
    display(validation_summary)
    validation_summary.to_csv(OUTPUT_ROOT / "validation_summary.csv", index=False)

    selected_row, ranked_models = select_model_from_macro_rows(
        validation_summary.query("project == 'MACRO_PROJECT_AVERAGE'")
    )
    model_selection = {
        "aggregation": SELECTION_AGGREGATION,
        "primary_metric": PRIMARY_SELECTION_METRIC,
        "tie_breakers": list(SELECTION_TIE_BREAKERS),
        "selected_model_name": selected_row["model_name"],
        "selected_model_revision": selected_row["model_revision"],
        "embedding_text_version": EMBEDDING_TEXT_VERSION,
        "encoding_mode": EMBEDDING_ENCODING_MODE,
        "query_prefix": QUERY_PREFIX,
        "document_prefix": DOCUMENT_PREFIX,
        "full_validation": MAX_VALIDATION_QUERIES is None,
        "scores": {
            metric: float(selected_row[metric])
            for metric in [PRIMARY_SELECTION_METRIC, *SELECTION_TIE_BREAKERS]
        },
    }
    (OUTPUT_ROOT / "model_selection.json").write_text(
        json.dumps(model_selection, indent=2), encoding="utf-8"
    )
    print("Frozen model-selection result:")
    print(json.dumps(model_selection, indent=2))

    phase1_validation_path = PHASE1_OUTPUT_ROOT / "validation_summary.csv"
    if phase1_validation_path.exists():
        print("Phase 1 lexical validation (reference):")
        display(pd.read_csv(phase1_validation_path))
else:
    print("FULL VALIDATION LOCKED: set RUN_FULL_VALIDATION=True after smoke passes.")


## 9. Final official test — keep sealed until validation is frozen

To run this section deliberately:

1. complete full validation with `MAX_VALIDATION_QUERIES = None`;
2. copy the model name and revision from `model_selection.json` into
   `SELECTED_TEST_MODEL_NAME` and `SELECTED_TEST_MODEL_REVISION`;
3. set `RUN_OFFICIAL_TEST = True`;
4. keep `MAX_TEST_QUERIES = None` for the publishable run.

Earlier official-test reports may become candidates for later test queries, as in
Phase 1, but every retrieved candidate must be strictly older than its query.


In [ ]:
test_results = {}
test_summary_with_macro = pd.DataFrame()

if RUN_OFFICIAL_TEST:
    assert SELECTED_TEST_MODEL_NAME and SELECTED_TEST_MODEL_REVISION, (
        "Set both selected model name and revision after full validation."
    )
    assert SELECTED_TEST_MODEL_NAME == EMBEDDING_MODEL_NAME, (
        "The official-test model must exactly match the model validated in this run."
    )
    assert SELECTED_TEST_MODEL_REVISION == EMBEDDING_MODEL_REVISION
    validation_artifact = OUTPUT_ROOT / "validation_summary.csv"
    selection_artifact = OUTPUT_ROOT / "model_selection.json"
    assert validation_artifact.exists(), (
        "Missing full validation artifact. Run Section 8 before opening test."
    )
    assert selection_artifact.exists(), "Missing frozen model-selection artifact."
    saved_validation = pd.read_csv(validation_artifact)
    saved_selection = json.loads(selection_artifact.read_text(encoding="utf-8"))
    assert saved_selection["full_validation"] is True
    assert saved_selection["aggregation"] == SELECTION_AGGREGATION
    assert saved_selection["primary_metric"] == PRIMARY_SELECTION_METRIC
    assert tuple(saved_selection["tie_breakers"]) == SELECTION_TIE_BREAKERS
    assert saved_selection["selected_model_name"] == SELECTED_TEST_MODEL_NAME
    assert saved_selection["selected_model_revision"] == SELECTED_TEST_MODEL_REVISION
    assert saved_selection["embedding_text_version"] == EMBEDDING_TEXT_VERSION
    assert saved_selection["encoding_mode"] == EMBEDDING_ENCODING_MODE
    assert saved_selection["query_prefix"] == QUERY_PREFIX
    assert saved_selection["document_prefix"] == DOCUMENT_PREFIX

    saved_project_validation = saved_validation[
        saved_validation["project"].isin(PROJECTS)
    ].copy()
    assert set(saved_project_validation["project"]) == set(PROJECTS)
    assert set(saved_validation["model_name"]) == {SELECTED_TEST_MODEL_NAME}
    assert set(saved_validation["model_revision"]) == {SELECTED_TEST_MODEL_REVISION}
    assert set(saved_validation["embedding_text_version"]) == {EMBEDDING_TEXT_VERSION}
    expected_validation_queries = {
        project: len(query_sets[project]["validation"])
        for project in PROJECTS
    }
    observed_validation_queries = dict(zip(
        saved_project_validation["project"],
        saved_project_validation["queries"].astype(int),
    ))
    assert observed_validation_queries == expected_validation_queries, (
        "Saved validation is sampled or stale; rerun the complete validation."
    )
    if MAX_VALIDATION_QUERIES is not None:
        raise ValueError("Official test requires a full, unsampled validation run.")

    test_summaries = []
    for project, data in datasets.items():
        print(f"\n=== OFFICIAL EMBEDDING TEST: {project} ===")
        representations = get_project_representations(project)
        results, summary = evaluate_dense_rolling(
            data=data,
            queries=query_sets[project]["test"],
            document_embeddings=representations["documents"],
            query_embeddings=representations["queries"],
            model_name=SELECTED_TEST_MODEL_NAME,
            model_revision=SELECTED_TEST_MODEL_REVISION,
            allowed_candidate_ids=(
                data["official_train_ids"] | data["official_test_ids"]
            ),
            max_queries=MAX_TEST_QUERIES,
            query_batch_size=EVAL_QUERY_BATCH_SIZE,
            device=EVAL_DEVICE,
        )
        test_results[project] = results
        test_summaries.append(summary)
        results.to_csv(OUTPUT_ROOT / f"{project.lower()}_test_query_results.csv", index=False)

    test_summary = pd.DataFrame(test_summaries)
    metric_columns = [
        *(f"hit@{k}" for k in TOP_KS),
        *(f"recall@{k}" for k in TOP_KS),
        "mrr@10", "ndcg@10",
    ]
    macro = {
        "project": "MACRO_PROJECT_AVERAGE",
        "representation": "embedding",
        "model_name": SELECTED_TEST_MODEL_NAME,
        "model_revision": SELECTED_TEST_MODEL_REVISION,
        "encoding_mode": EMBEDDING_ENCODING_MODE,
        "embedding_text_version": EMBEDDING_TEXT_VERSION,
        "queries": int(test_summary["queries"].sum()),
        **{column: test_summary[column].mean() for column in metric_columns},
    }
    test_summary_with_macro = pd.concat(
        [test_summary, pd.DataFrame([macro])], ignore_index=True
    )
    display(test_summary_with_macro)
    test_summary_with_macro.to_csv(OUTPUT_ROOT / "test_summary.csv", index=False)

    phase1_test_path = PHASE1_OUTPUT_ROOT / "test_summary.csv"
    if phase1_test_path.exists():
        print("Phase 1 lexical test (reference):")
        display(pd.read_csv(phase1_test_path))
else:
    print("OFFICIAL TEST SEALED: leave RUN_OFFICIAL_TEST=False until selection is frozen.")


## 10. Leakage checks and run manifest

The manifest distinguishes smoke, full validation, and official test. A run is
publishable only when official test was explicitly enabled and left unsampled.


In [ ]:
for project, results in test_results.items():
    data = datasets[project]
    assert results["query_id"].is_unique
    for row in results.itertuples(index=False):
        query_time = data["id_to_time"][row.query_id]
        assert all(
            data["id_to_time"][retrieved_id] < query_time
            for retrieved_id in row.retrieved_ids.split(";")
            if retrieved_id
        )

manifest = {
    "phase": 2,
    "protocol": (
        "frozen zero-shot embeddings; explicit split mask; "
        "rolling strictly-older candidate history"
    ),
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "embedding_model_revision": EMBEDDING_MODEL_REVISION,
    "embedding_encoding_mode": EMBEDDING_ENCODING_MODE,
    "query_prefix": QUERY_PREFIX,
    "document_prefix": DOCUMENT_PREFIX,
    "selected_test_model_name": SELECTED_TEST_MODEL_NAME,
    "selected_test_model_revision": SELECTED_TEST_MODEL_REVISION,
    "selection_rule": {
        "aggregation": SELECTION_AGGREGATION,
        "primary_metric": PRIMARY_SELECTION_METRIC,
        "tie_breakers": list(SELECTION_TIE_BREAKERS),
    },
    "embedding_text_version": EMBEDDING_TEXT_VERSION,
    "truncation_warning_threshold_pct": TRUNCATION_WARNING_THRESHOLD_PCT,
    "top_ks": list(TOP_KS),
    "run_smoke_test": RUN_SMOKE_TEST,
    "smoke_max_queries": SMOKE_MAX_QUERIES,
    "run_full_validation": RUN_FULL_VALIDATION,
    "max_validation_queries": MAX_VALIDATION_QUERIES,
    "run_official_test": RUN_OFFICIAL_TEST,
    "max_test_queries": MAX_TEST_QUERIES,
    "full_publishable_run": (
        RUN_OFFICIAL_TEST
        and MAX_VALIDATION_QUERIES is None
        and MAX_TEST_QUERIES is None
        and SELECTED_TEST_MODEL_NAME == EMBEDDING_MODEL_NAME
        and SELECTED_TEST_MODEL_REVISION == EMBEDDING_MODEL_REVISION
    ),
    "random_seed": RANDOM_SEED,
    "data_root": str(DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "environment": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "sentence_transformers": sentence_transformers.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    "embedding_cache_metadata": {
        f"{project}::{role}": metadata
        for (project, role), metadata in embedding_metadata.items()
    },
    "split_chronology": split_chronology.assign(
        train_min=split_chronology["train_min"].astype(str),
        train_max=split_chronology["train_max"].astype(str),
        test_min=split_chronology["test_min"].astype(str),
        test_max=split_chronology["test_max"].astype(str),
    ).to_dict(orient="records"),
    "projects": {
        project: {
            "issues": len(data["issues"]),
            "official_train": len(data["train_labels"]),
            "official_test": len(data["test_labels"]),
            "evaluable_validation_queries": len(query_sets[project]["validation"]),
            "evaluable_test_queries": len(query_sets[project]["test"]),
        }
        for project, data in datasets.items()
    },
}

(OUTPUT_ROOT / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print(json.dumps(manifest, indent=2))
print("\nLEAKAGE CHECKS: PASS")
print("Artifacts:", OUTPUT_ROOT)


## Interpretation and reporting

- `Hit@K`: at least one known historical duplicate appears in Top-K.
- `Recall@K`: fraction of all known historical duplicates retrieved in Top-K.
- `MRR@10`: reciprocal rank of the first relevant result, truncated at 10.
- `nDCG@10`: rewards placing multiple relevant reports near the top.

Report Phase 2 as a **frozen zero-shot embedding baseline**, not a trained model.
Also disclose the model ID, package versions, embedding input policy, estimated
token truncation rate, temporal protocol, and whether the run was sampled.

Do not report the smoke metrics as final results. Use the complete validation to
freeze the model, then execute the official test exactly once and compare its
saved summary with Phase 1 TF-IDF.
